# Assignment 1 — LLM evaluation (product descriptions)

Build and evaluate **short product descriptions** from structured product data: define quality **before** generating at scale, run a **baseline**, manually score a sample, try **targeted improvements**, then compare **human** ratings to an **LLM judge** with structured output.

This notebook (v2) targets the **Nebius** OpenAI-compatible API.

**Project context, conda setup, and dataset noise notes:** see **`README.md`** in this folder.

What this notebook helps you do:
- define a rubric
- generate product descriptions
- save `assignment_01.xlsx`
- manually evaluate a sample
- run improvement experiments
- build an LLM judge with structured output
- compare judge vs human agreement

Replace placeholder values where needed.

## 0. Colab quick start

If you are new to notebooks, use this mental model:

- **Markdown cell** = explanation / notes
- **Code cell** = Python you can run with Shift+Enter
- Variables stay in memory **only after you run the cell**
- Run cells from top to bottom
- If runtime resets, run everything again from the start

Suggested workflow:
1. Upload this notebook to Colab
2. Upload the dataset CSV to Colab, or mount Drive
3. Add your Nebius API key
4. Run baseline generation
5. Save the Excel file
6. Fill the manual evaluation rows
7. Run judge evaluation

In [ ]:
# If needed in Colab, uncomment:
# !pip install -q pandas openpyxl pydantic openai tqdm

## 0-2. Local setup (Cursor, VS Code, or Jupyter Lab)

If you work on your machine (not Colab):

1. Create a Python **3.11+** environment and install `pandas`, `openpyxl`, `openai`, `pydantic`, `tqdm`, plus `jupyter` / `ipykernel` if you use the notebook UI.
2. Put the course CSV files under **`data/`** next to this notebook (see **README.md** for exact filenames).



### Step 1: Create a project folder

Make one folder for the whole assignment, for example:

```text
module1_LLM_Evaluation_Asignment01/
```

Put inside it docs/, scripts/, data/, and models/ folders:

* the dataset CSV in data/
* your notebook
* optional notes file in docs/

### Step 2: Create your Python environment
```bash
conda create -n assignment01 python=3.11 -y
conda activate assignment01
conda install pandas openpyxl tqdm jupyter ipykernel nbformat -y
pip install pydantic openai
```

### Step 3: Register the environment as a Jupyter kernel

This is what makes your environment show up inside the notebook UI:

```bash
python -m ipykernel install --user --name assignment01 --display-name "Python (assignment01)"
```

### Step 4: Open the folder

Then open your notebook or create a new `.ipynb`.

### Step 5: Select the kernel

In the notebook UI, choose:

```text
Python (assignment01)
```

## 1. Setup and imports

**Environment:** Python 3.11+ with `pandas`, `openpyxl` (Excel), `openai` (API), `pydantic` (structured schemas), `tqdm` (progress). The notebook kernel must be the environment where these packages are installed.

| Area | Why it matters |
|------|----------------|
| `pandas` | Load CSV, inspect rows, build the Excel artifact. |
| `openpyxl` | Write `.xlsx` for submission. |
| `openai` | OpenAI-compatible client for Nebius chat/completions. |
| `pydantic` | Enforce **structured** judge outputs (scores + short rationale). |

**Secrets:** set `NEBIUS_API_KEY` in the environment (configured in section 3). Do not put keys in the notebook.

> **Week 1 — context engineering:** Later prompts need the **right fields** in a consistent order; treat the dataframe as the single source of truth for grounding.

> **Week 2 — evaluation-driven development:** Keep comparing runs against the **same rubric** and inspection sample.

### Imports

Run the next cell to import libraries.

In [1]:
import os
import re
import time
import math
import json
from typing import Literal, Optional

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
from pydantic import BaseModel, Field

from openai import OpenAI

## 2. Load dataset

**Source:** CSV of products with columns such as `product_name`, `Product_attribute_list`, `material`, `warranty`.

**Local / Cursor:** use `data/01_agents_Assignment_01_product_dataset.csv` (path relative to this notebook when the working directory is this folder).

**Colab:** open the folder icon, upload `01_agents_Assignment_01_product_dataset.csv`, and set `DATASET_PATH` to that file (or `/content/...`).

**Before any generation:**

1. Confirm **row count** and **column names** match expectations.
2. Spot-check a few rows for **missing values**, odd encodings, or **multi-value** fields.
3. Note which columns are **safe to quote** verbatim vs need **paraphrase** (e.g. internal codes).

**Grounding checklist**

- Which fields are **authoritative** for factual claims (material, size, weight)?
- Which fields are **marketing-ish** and should not be treated as independent verification?
- Where the data is **sparse**, the model should **not** fill gaps with guesses.

This inspection feeds **Task 1**: each rubric criterion should be checkable against **these** columns.

> **Week 2 — same evaluation slice:** Keep a fixed **inspection sample** (row indices or product IDs) when comparing baseline vs improved prompts and when aligning the judge with humans.

**Optional second file:** the challenging CSV is loaded in the following cell for comparison only. Generation below uses **`df_products`** from the first path — change `DATASET_PATH` if you want to run the pipeline on the challenging file instead.

For **noise, hallucination hotspots, and CSV comparison**, see **`README.md`**.

In [2]:
# Default: standard course CSV (local). On Colab, point this at your uploaded file.
DATASET_PATH = "data/01_agents_Assignment_01_product_dataset.csv"  # change if needed

df_products = pd.read_csv(DATASET_PATH)

print(df_products.shape)
print(df_products.columns.tolist())
display(df_products.head())
display(df_products.isna().mean().sort_values(ascending=False).head(10))

(50, 4)
['product_name', 'Product_attribute_list', 'material', 'warranty']


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty


product_name              0.0
Product_attribute_list    0.0
material                  0.0
warranty                  0.0
dtype: float64

### Optional: challenging dataset (inspection only)

Same schema as the standard file; rows emphasize negations and tricky warranties. Load into **`df_products_challenging`** for side-by-side exploration. **`df_products`** (above) is what the rest of this notebook uses for generation unless you change `DATASET_PATH` and re-run.

In [3]:
DATASET_CHALLENGING_PATH = "data/Assignment_01_product_dataset_CHALLENGING.csv"  # change if needed

df_products_challenging = pd.read_csv(DATASET_CHALLENGING_PATH)

print(df_products_challenging.shape)
print(df_products_challenging.columns.tolist())
display(df_products_challenging.head())
display(df_products_challenging.isna().mean().sort_values(ascending=False).head(10))

(48, 4)
['product_name', 'Product_attribute_list', 'material', 'warranty']


,product_name,Product_attribute_list,material,warranty
0,EcoBlend Pro Coffee Maker,"features: 12-cup capacity, programmable timer,...","glass carafe, BPA-free plastic housing",90-day limited warranty
1,TechNova Wireless Earbuds,"features: Bluetooth 5.0, 6-hour battery per ch...","silicone tips, plastic charging case","1-year warranty on electronics, 30-day warrant..."
2,FitTrack Smart Scale,"features: measures weight only, does NOT measu...",tempered glass platform with plastic base,6-month limited warranty
3,UltraGrip Kitchen Knife Set,"features: 5-piece set includes chef, paring, b...",high-carbon stainless steel blades with polyme...,lifetime warranty on manufacturing defects (do...
4,AquaPure Water Filter Pitcher,features: reduces chlorine taste and odor; doe...,BPA-free plastic pitcher with activated carbon...,no warranty on pitcher; filters sold separately


product_name              0.0
Product_attribute_list    0.0
material                  0.0
warranty                  0.0
dtype: float64

## 3. Configure API client

Nebius uses an OpenAI-compatible API.

**Environment variables** (shell export or a `.env` file — see next cell):

- `NEBIUS_API_KEY` — required for API calls.
- `LLM_BASE_URL` (or `NEBIUS_BASE_URL` / `BASE_URL`) — optional; defaults to Nebius Studio if unset.
- `LLM_MODEL` or `LLM_GENERATION_MODEL` — optional override for the generation model string.

The next cell loads `.env` from the current working directory, its parent, or grandparent (so a repo-root `.env` still works when the notebook lives in a subfolder). Install **`python-dotenv`** locally: `pip install python-dotenv`.

**Course models** (if your rubric requires them): `meta-llama/Meta-Llama-3.1-8B-Instruct` or `google/gemma-2-9b-it`. If you use Token Factory, set `LLM_BASE_URL` and the model id to match your dashboard.

In [4]:
# Safer option in Colab:
# from getpass import getpass
# os.environ["NEBIUS_API_KEY"] = getpass("Enter Nebius API key: ")

from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# Find .env next to the notebook cwd, or one/two levels up (e.g. repo root above module1_.../)
if load_dotenv is not None:
    _here = Path.cwd().resolve()
    for _env in (_here / ".env", _here.parent / ".env", _here.parent.parent / ".env"):
        if _env.is_file():
            load_dotenv(_env)
            break
    else:
        load_dotenv()
else:
    print("Tip: pip install python-dotenv to load NEBIUS_API_KEY from a .env file.")

_default_base = "https://api.studio.nebius.ai/v1/"
BASE_URL = (
    os.getenv("LLM_BASE_URL")
    or os.getenv("NEBIUS_BASE_URL")
    or os.getenv("BASE_URL")
    or _default_base
).rstrip("/") + "/"

API_KEY = (os.getenv("NEBIUS_API_KEY") or "").strip().strip('"').strip("'")

# Optional: set LLM_MODEL or LLM_GENERATION_MODEL in .env to override the default below.
GENERATION_MODEL = (
    os.getenv("LLM_GENERATION_MODEL")
    or os.getenv("LLM_MODEL")
    or "meta-llama/Meta-Llama-3.1-8B-Instruct"
)

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)

if not API_KEY:
    print("WARNING: NEBIUS_API_KEY is empty. Set it in .env or export it before starting Jupyter.")
else:
    print("Client configured (API key present).")
print(f"BASE_URL={BASE_URL!r}")
print(f"GENERATION_MODEL={GENERATION_MODEL!r}")

Client configured.


## 4. Rubric

Edit this section to match the rubric you want to submit.

Recommended default:
- strict on grounding
- strict on length
- grammar bad = fail

In [ ]:
RUBRIC = {
    "fluency": {
        "good": "Natural, smooth, easy to read, no awkward phrasing.",
        "ok": "Understandable but slightly awkward, repetitive, or stiff.",
        "bad": "Hard to read, unnatural, choppy, or confusing."
    },
    "grammar": {
        "good": "No grammar, spelling, or punctuation issues.",
        "ok": "1-2 minor issues that do not harm understanding.",
        "bad": "Multiple or noticeable grammar, spelling, or punctuation issues."
    },
    "tone": {
        "good": "Friendly, credible, persuasive sales tone without hype.",
        "ok": "Mostly appropriate but bland, generic, or slightly too promotional.",
        "bad": "Robotic, off-brand, exaggerated, pushy, or not persuasive."
    },
    "length": {
        "good": "50-90 words.",
        "ok": "40-49 words or 91-110 words.",
        "bad": "Under 40 or over 110 words."
    },
    "grounding": {
        "good": "All claims are supported by provided product data.",
        "ok": "Minor inference or generic phrasing, but no clear invented facts.",
        "bad": "Invents unsupported features, specs, benefits, ratings, certifications, or use cases."
    }
}

PASS_RULE = "Pass if grounding == good, grammar != bad, length != bad, and at least 4 criteria are good."
AUTO_FAIL_RULES = [
    "grounding != good",
    "grammar == bad",
    "length == bad",
]

RUBRIC

## 5. Baseline prompt

This is the first prompt for Task 2.

In [ ]:
SYSTEM_PROMPT_BASELINE = """
You are a careful e-commerce copywriter.

Write a persuasive product description in 50-90 words using only the provided product information.
Requirements:
- Keep the tone friendly, credible, and sales-oriented
- Use natural, fluent English
- Do not invent features, specs, ratings, materials, certifications, or use cases
- Mention concrete details from the input when possible
- If information is limited, stay concise rather than making assumptions
- Output only the final description
""".strip()

def build_user_prompt(row: pd.Series) -> str:
    return f"""Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

Write the product description now."""

## 6. Helper functions

In [ ]:
def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def compute_final_score(row: pd.Series) -> str:
    criteria = ["fluency", "grammar", "tone", "length", "grounding"]
    good_count = sum(row.get(c) == "good" for c in criteria)

    if row.get("grounding") != "good":
        return "fail"
    if row.get("grammar") == "bad":
        return "fail"
    if row.get("length") == "bad":
        return "fail"
    if good_count >= 4:
        return "pass"
    return "fail"

In [ ]:
def generate_description_for_row(row: pd.Series, system_prompt: str, model: str, temperature: float = 0.3, max_tokens: int = 140):
    user_prompt = build_user_prompt(row)

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    latency_ms = round((time.perf_counter() - t0) * 1000, 2)

    text = response.choices[0].message.content.strip()

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "prompt_tokens", None)
    output_tokens = getattr(usage, "completion_tokens", None)

    return {
        "generated_description": text,
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

## 7. Baseline generation run

This creates the main dataframe for Task 2 and saves `assignment_01.xlsx`.

Run on all rows when ready.

In [ ]:
results = []

for _, row in tqdm(df_products.iterrows(), total=len(df_products)):
    out = generate_description_for_row(
        row=row,
        system_prompt=SYSTEM_PROMPT_BASELINE,
        model=GENERATION_MODEL,
        temperature=0.3,
        max_tokens=140,
    )
    results.append(out)

df_assignment = pd.concat([df_products.reset_index(drop=True), pd.DataFrame(results)], axis=1)

for col in ["fluency", "grammar", "tone", "length", "grounding", "final_score"]:
    df_assignment[col] = ""

df_assignment.head()

In [ ]:
OUTPUT_XLSX = "assignment_01.xlsx"
df_assignment.to_excel(OUTPUT_XLSX, index=False)
print(f"Saved {OUTPUT_XLSX}")

## 8. Cost calculation

Fill in your model pricing.

Use **USD per 1M tokens** and the code will convert to per-row cost.

Example:
- input price: 0.20
- output price: 0.60

Replace placeholders with the actual course pricing for your selected model.

In [ ]:
INPUT_PRICE_PER_1M = 0.0   # TODO
OUTPUT_PRICE_PER_1M = 0.0  # TODO

def compute_cost_usd(input_tokens: Optional[int], output_tokens: Optional[int]) -> Optional[float]:
    if input_tokens is None or output_tokens is None:
        return None
    return round((input_tokens / 1_000_000) * INPUT_PRICE_PER_1M + (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M, 8)

df_assignment["cost_usd"] = df_assignment.apply(
    lambda r: compute_cost_usd(r["input_tokens"], r["output_tokens"]),
    axis=1,
)

df_assignment[["product_name", "input_tokens", "output_tokens", "cost_usd"]].head()

## 9. Manual evaluation sample

Pick 10–15 rows for human evaluation.

You can change the sampled indices manually if you want category coverage.

In [ ]:
manual_eval_indices = [0, 1, 2, 3, 4, 10, 15, 20, 25, 30, 35, 40]
df_manual = df_assignment.loc[manual_eval_indices, [
    "product_name",
    "Product_attribute_list",
    "material",
    "warranty",
    "generated_description"
]].copy()

df_manual

After reviewing the descriptions manually, write your ratings below.

Allowed values: `good`, `ok`, `bad`

In [ ]:
# TODO: fill manually after reading the selected descriptions
manual_scores = {
    # index: {"fluency": "good", "grammar": "good", "tone": "ok", "length": "good", "grounding": "good"},
}

for idx, scores in manual_scores.items():
    for k, v in scores.items():
        df_assignment.loc[idx, k] = v

for idx in manual_scores:
    df_assignment.loc[idx, "final_score"] = compute_final_score(df_assignment.loc[idx])

df_assignment.loc[manual_eval_indices, ["product_name", "fluency", "grammar", "tone", "length", "grounding", "final_score"]]

## 10. Baseline analysis

This helps you write the Task 3 discussion.

In [ ]:
human_eval_subset = df_assignment.loc[
    df_assignment.index.isin(manual_eval_indices) & (df_assignment["final_score"] != ""),
    ["fluency", "grammar", "tone", "length", "grounding", "final_score"]
].copy()

for col in ["fluency", "grammar", "tone", "length", "grounding"]:
    print("\n", col.upper())
    print(human_eval_subset[col].value_counts(dropna=False))

## 11. Improvement experiment section

Duplicate this block for each experiment.

Examples to try:
- stricter grounding prompt
- lower temperature
- second-pass shortening
- different generation model

In [ ]:
SYSTEM_PROMPT_EXPERIMENT_1 = """
You are a careful e-commerce copywriter.

Write one product description in 50-90 words.
Use only the facts provided.
Do not add claims about performance, comfort, durability, compatibility, premium quality, or user benefits unless directly supported by the input.
Prefer concrete product facts over generic marketing language.
Keep the tone friendly and credible.
Output only the description.
""".strip()

EXPERIMENT_MODEL = GENERATION_MODEL
EXPERIMENT_NAME = "exp_1_stricter_grounding"

exp_results = []
for _, row in tqdm(df_products.iterrows(), total=len(df_products)):
    out = generate_description_for_row(
        row=row,
        system_prompt=SYSTEM_PROMPT_EXPERIMENT_1,
        model=EXPERIMENT_MODEL,
        temperature=0.2,
        max_tokens=140,
    )
    exp_results.append(out)

df_exp1 = pd.concat([df_products.reset_index(drop=True), pd.DataFrame(exp_results)], axis=1)
df_exp1.head()

Re-evaluate the same human sample for the experiment so comparison is fair.

In [ ]:
# TODO: fill manually for the same sample indices
experiment_1_manual_scores = {
    # index: {"fluency": "good", "grammar": "good", "tone": "ok", "length": "good", "grounding": "good"},
}

for col in ["fluency", "grammar", "tone", "length", "grounding", "final_score"]:
    df_exp1[col] = ""

for idx, scores in experiment_1_manual_scores.items():
    for k, v in scores.items():
        df_exp1.loc[idx, k] = v
    df_exp1.loc[idx, "final_score"] = compute_final_score(df_exp1.loc[idx])

df_exp1.loc[manual_eval_indices, ["product_name", "fluency", "grammar", "tone", "length", "grounding", "final_score"]]

## 12. Judge model

Start with the model you did **not** use for generation.
If it is weak as a judge, switch to a larger one and document why.

In [ ]:
JUDGE_MODEL = "google/gemma-2-9b-it"  # change if needed

In [ ]:
class CriterionScore(BaseModel):
    explanation: str = Field(description="Short reasoning for the verdict based on the rubric and the product input.")
    verdict: Literal["good", "ok", "bad"]

class JudgeOutput(BaseModel):
    fluency: CriterionScore
    grammar: CriterionScore
    tone: CriterionScore
    length: CriterionScore
    grounding: CriterionScore

In [ ]:
def rubric_text() -> str:
    parts = []
    for criterion, defs in RUBRIC.items():
        parts.append(
            f"{criterion.capitalize()}:\n"
            f"- good: {defs['good']}\n"
            f"- ok: {defs['ok']}\n"
            f"- bad: {defs['bad']}"
        )
    return "\n\n".join(parts)

JUDGE_PROMPT = f"""
You are a strict evaluation judge for e-commerce product descriptions.

Use the rubric exactly as written below.

{rubric_text()}

Evaluation rules:
- Evaluate only fluency, grammar, tone, length, and grounding
- Be strict about grounding
- Do not reward unnecessary verbosity
- Use the product input as the source of truth
- Explanation must come before verdict
- Return structured output only
""".strip()

print(JUDGE_PROMPT[:2000])

### Why explanation before verdict?

Use this idea in your report:

Because autoregressive models generate left-to-right, asking for reasoning before the label helps the model inspect evidence first and then commit to a verdict, instead of picking a label early and rationalizing it afterward.

In [ ]:
def judge_one(row: pd.Series, model: str = JUDGE_MODEL):
    user_prompt = f"""Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

Description to evaluate:
{row['generated_description']}"""

    response = client.beta.chat.completions.parse(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=JudgeOutput,
    )
    return response.choices[0].message.parsed

## 13. Judge sanity check on 5 products

In [ ]:
sanity_indices = manual_eval_indices[:5]
sanity_outputs = {}

for idx in sanity_indices:
    sanity_outputs[idx] = judge_one(df_assignment.loc[idx])

sanity_outputs

Review the 5 outputs manually before doing a full run.
If needed, adjust `JUDGE_PROMPT`.

## 14. Full judge run

In [ ]:
judge_rows = []

for idx, row in tqdm(df_assignment.iterrows(), total=len(df_assignment)):
    judged = judge_one(row)
    judge_rows.append({
        "judge_fluency_explanation": judged.fluency.explanation,
        "judge_fluency": judged.fluency.verdict,
        "judge_grammar_explanation": judged.grammar.explanation,
        "judge_grammar": judged.grammar.verdict,
        "judge_tone_explanation": judged.tone.explanation,
        "judge_tone": judged.tone.verdict,
        "judge_length_explanation": judged.length.explanation,
        "judge_length": judged.length.verdict,
        "judge_grounding_explanation": judged.grounding.explanation,
        "judge_grounding": judged.grounding.verdict,
    })

df_judge = pd.concat([df_assignment.reset_index(drop=True), pd.DataFrame(judge_rows)], axis=1)
df_judge.head()

In [ ]:
def compute_final_score_from_prefix(row: pd.Series, prefix: str = "") -> str:
    mapping = {
        "fluency": row[f"{prefix}fluency"],
        "grammar": row[f"{prefix}grammar"],
        "tone": row[f"{prefix}tone"],
        "length": row[f"{prefix}length"],
        "grounding": row[f"{prefix}grounding"],
    }
    good_count = sum(v == "good" for v in mapping.values())

    if mapping["grounding"] != "good":
        return "fail"
    if mapping["grammar"] == "bad":
        return "fail"
    if mapping["length"] == "bad":
        return "fail"
    if good_count >= 4:
        return "pass"
    return "fail"

df_judge["judge_final_score"] = df_judge.apply(lambda r: compute_final_score_from_prefix(r, prefix="judge_"), axis=1)
df_judge[["product_name", "judge_fluency", "judge_grammar", "judge_tone", "judge_length", "judge_grounding", "judge_final_score"]].head()

## 15. Agreement with human evaluation

In [ ]:
agreement = {}

for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
    subset = df_judge.loc[manual_eval_indices, [criterion, f"judge_{criterion}"]].copy()
    subset = subset[(subset[criterion] != "") & subset[f"judge_{criterion}"].notna()]
    if len(subset) == 0:
        agreement[criterion] = None
    else:
        agreement[criterion] = (subset[criterion] == subset[f"judge_{criterion}"]).mean()

agreement

## 16. Criterion-by-criterion judging

This repeats judging one criterion at a time.
You can compare whether isolated judging improves agreement.

In [ ]:
class SingleCriterionOutput(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]

def judge_single_criterion(row: pd.Series, criterion: str, model: str = JUDGE_MODEL):
    system_prompt = f"""
You are a strict evaluation judge for e-commerce product descriptions.

Evaluate only this criterion: {criterion}

Rubric:
{criterion.capitalize()}:
- good: {RUBRIC[criterion]['good']}
- ok: {RUBRIC[criterion]['ok']}
- bad: {RUBRIC[criterion]['bad']}

Rules:
- Use the product input as the source of truth
- For grounding, be strict
- Do not evaluate any other criterion
- Explanation must come before verdict
- Return structured output only
""".strip()

    user_prompt = f"""Product name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

Description to evaluate:
{row['generated_description']}"""

    response = client.beta.chat.completions.parse(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=SingleCriterionOutput,
    )
    return response.choices[0].message.parsed

Run the block below only if you want the full per-criterion experiment.
It is slower because it makes many more API calls.

In [ ]:
# Example: run only on the manual sample first
single_criterion_rows = []

for idx in tqdm(manual_eval_indices):
    row = df_assignment.loc[idx]
    out = {"index": idx}
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        judged = judge_single_criterion(row, criterion)
        out[f"single_{criterion}_explanation"] = judged.explanation
        out[f"single_{criterion}"] = judged.verdict
    single_criterion_rows.append(out)

df_single = pd.DataFrame(single_criterion_rows)
df_single

In [ ]:
single_agreement = {}

merged = df_assignment.loc[manual_eval_indices, ["fluency", "grammar", "tone", "length", "grounding"]].copy()
merged["index"] = merged.index
merged = merged.merge(df_single, on="index", how="left")

for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
    subset = merged[[criterion, f"single_{criterion}"]].copy()
    subset = subset[(subset[criterion] != "") & subset[f"single_{criterion}"].notna()]
    if len(subset) == 0:
        single_agreement[criterion] = None
    else:
        single_agreement[criterion] = (subset[criterion] == subset[f"single_{criterion}"]).mean()

single_agreement

## 17. Save final outputs

In [ ]:
df_judge.to_excel("assignment_01_with_judge.xlsx", index=False)
print("Saved assignment_01_with_judge.xlsx")

## 18. Suggested report structure

Use this as your writeup outline:

1. Rubric
2. Pass/fail rule and automatic no-go rules
3. Baseline generation setup
4. Human evaluation sample and results
5. Baseline analysis: best/worst criteria and failure buckets
6. Improvement experiments
7. Judge model design
8. Why explanation-before-verdict matters
9. Judge sanity check
10. Full judge results
11. Agreement with human evaluation
12. Criterion-by-criterion judge comparison
13. Human vs judge trade-offs
14. Production recommendation